# 10 The PRESAGE model
Streams: **z_txn** (MLP on payment/velocity/beneficiary features), **z_beh** (GRU over the sender's previous payments), **z_graph** (temporal heterogeneous encoder), **z_mule** (mule module on the beneficiary embedding). Fusion: concat, gated or attention. Heads: APP fraud (A), beneficiary mule (B), fraud-network membership (C); uncertainty (D) by MC-dropout.

In [1]:
import sys, pathlib, json, warnings; warnings.filterwarnings('ignore')
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
plt.rcParams.update({'figure.dpi': 100, 'axes.spines.top': False, 'axes.spines.right': False})
def table(name):
    p = ROOT / 'reports' / 'tables' / name
    return pd.read_csv(p) if p.exists() else None

In [2]:
from presage.models.presage import Presage, PresageConfig
import torch
cfg = PresageConfig(); m = Presage(cfg, d_node=11, d_tab=70, d_seq=11, n_seq=11, ben_idx=list(range(10)))
print(sum(p.numel() for p in m.parameters()), 'parameters'); print(cfg)

225677 parameters
PresageConfig(h=64, use_txn=True, use_beh=True, use_graph=True, use_mule=True, temporal=True, multitask=True, fusion='gated', seq_kind='gru', dropout=0.15, lam=(1.0, 0.5, 0.3), recent_days=7.0)


In [3]:
from presage.data.prepare import prepare
D = prepare(); A, snaps = D['A'], D['snaps']
k = 20; idx = np.where(D['snap_of']==k)[0][:256]
mdl = Presage(cfg, snaps[0].x.shape[1], A['tab'].shape[1], A['seq'].shape[2], A['seq'].shape[1], D['ben_idx'])
out = mdl(snaps[k], torch.tensor(A['src'][idx]), torch.tensor(A['dst'][idx]), torch.tensor(A['tab'][idx]), torch.tensor(A['seq'][idx]), torch.tensor(A['mask'][idx]))
print({k_: (tuple(v.shape) if v is not None else None) for k_, v in out.items()}); print('gate weights (txn,beh,graph,mule) mean:', out['weights'].mean(0).detach().numpy().round(3), '(untrained)')

19:18:04 INFO snapshots: built 31 snapshots (15262 accounts, 17841 devices, 8000 ips)


{'app': (256,), 'mule': (256,), 'net': (256,), 'weights': (256, 4)}
gate weights (txn,beh,graph,mule) mean: [0.231 0.273 0.268 0.228] (untrained)


Training uses chronological snapshots: for each snapshot the graph is encoded once, the account memory is advanced, and payments in that window are scored and back-propagated (negatives subsampled with importance weights; positives kept). Validation PR-AUC picks the epoch. See `scripts/run_presage.py`.

In [4]:
a = pd.concat([table(f) for f in ['presage_full_s42.csv','presage_concat_attention_transformer_seq_s42.csv'] if table(f) is not None]); a.round(3)

,variant,pr_auc,pr_auc_lo,pr_auc_hi,roc_auc,recall,precision,f1,f2,mcc,fpr,cost_costthr,prevented_ratio,mule_pr_auc,train_s,latency_ms,n_params
0,full,0.919,0.870,0.951,0.986,0.858,0.953,0.903,0.876,0.904,0.000,8315.226,0.975,0.996,439.078,0.344,229517
0,concat,0.982,0.963,0.990,1.000,0.894,0.955,0.923,0.905,0.923,0.000,3694.067,0.998,0.992,429.984,0.356,261321
1,attention,0.943,0.900,0.964,0.999,0.835,0.922,0.876,0.851,0.877,0.000,6465.128,0.984,0.993,444.126,0.409,232713
2,transformer_seq,0.406,0.316,0.492,0.978,0.726,0.280,0.404,0.551,0.447,0.009,21218.926,0.951,0.992,522.164,0.609,283149
